In [294]:
import pandas as pd
import numpy as np

# This data set is from the larger dataset of "Patent Examination Research Dataset (PatEx) for Academia and Researchers" of 'USPTO Bulk Data Directory'
application_data = pd.read_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/examiner/application_data.csv")

/var/folders/gn/x5brd2hs44x9lc3g72_l7b680000gn/T/ipykernel_5731/3007494631.py:5: DtypeWarning: Columns (0,3,4,6,12,13,14,16,17,18,21) have mixed types. Specify dtype option on import or set low_memory=False.
  application_data = pd.read_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/examiner/application_data.csv")


In [295]:
# Lets see what columns we have
application_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14100378 entries, 0 to 14100377
Data columns (total 22 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   application_number          object 
 1   filing_date                 object 
 2   application_invention_type  object 
 3   examiner_full_name          object 
 4   examiner_art_unit           object 
 5   uspc_class                  object 
 6   uspc_subclass               object 
 7   confirm_number              float64
 8   atty_docket_number          object 
 9   appl_status_desc            object 
 10  appl_status_date            object 
 11  file_location               object 
 12  file_location_date          object 
 13  earliest_pgpub_number       object 
 14  earliest_pgpub_date         object 
 15  wipo_pub_number             float64
 16  wipo_pub_date               object 
 17  patent_number               object 
 18  patent_issue_date           object 
 19  invention_title    

In [296]:
application_data["wipo_pub_number"].value_counts(dropna=False)
# "The World Intellectual Property Organization (WIPO) is the United Nations agency that serves the world’s innovators and creators". Mostly NaN.

# File location is where the patent file is stored. Not required for my work.
application_data["file_location"].value_counts()

file_location
ELECTRONIC                                                                          9561928
FILE REPOSITORY (FRANCONIA)                                                         3293741
NO LONGER FORMER                                                                      16043
At National Archives - Contact Records Management at RecordsManagement@uspto.gov      13932
FORMER GROUP 2300                                                                     13924
                                                                                     ...   
TC 2600 HSLIE, KNX-08D51, 571-272-7268                                                    1
TC 3700 LDRC TEAM 2, RND-6TH FLOOR                                                        1
RSQ/6TH FLOOR                                                                             1
TC 1600 - GROUP ART UNIT 1654                                                             1
TC 3600 PAPER MATCHING, PK-5 3B05                                 

In [297]:
# Alongside the tech center number, the uspc class also gives us information about the technology of the patent.
# However they are too detailed and subject to reclassification.
# I want to drop the class types of 'Design' and 'Plant' and only keep the Utility classes.
s = application_data["uspc_class"].astype("string")
mask = s.str.startswith("PLT", na=False)

application_data = application_data[~mask]

In [298]:
s = application_data["uspc_class"].astype("string")
mask2 = s.str.startswith("D", na=False)

application_data = application_data[~mask2]

In [299]:
# I drop the columns I am not interested in
application_data2 = application_data.drop(columns=[
    "application_number",
    "application_invention_type", "uspc_class", "uspc_subclass","confirm_number",
    "atty_docket_number", "file_location","file_location_date", "appl_status_date",
    "earliest_pgpub_number","earliest_pgpub_date", "wipo_pub_number","wipo_pub_date",
    "invention_title", "small_entity_indicator", "aia_first_to_file"
])

In [300]:
# I convert float values to integers and strings.
application_data2["examiner_art_unit"] = pd.to_numeric(application_data2["examiner_art_unit"], errors="coerce").astype("Int64")
application_data2["appl_status_desc"] = application_data2["appl_status_desc"].astype("string")

# I want to create a new column for examiner tech unit and I will use the examiner art unit for this.
#application_data2["examiner_tech_center_number"] = (application_data2["examiner_art_unit"] / 100).astype("Int64")*100

In [301]:
# I will filter the dates now
application_data2["patent_issue_date"] = pd.to_datetime(application_data2["patent_issue_date"], errors="coerce")
application_data2["filing_date"] = pd.to_datetime(application_data2["filing_date"], errors="coerce")

start = pd.Timestamp("1997-12-01")
end = pd.Timestamp("2023-12-31")

m_filing = application_data2["filing_date"].between(start, end, inclusive="both")

psd = application_data2["patent_issue_date"]
m_issue = psd.isna() | psd.between(start, end, inclusive="both")

application_data3 = application_data2[m_filing & m_issue]

In [302]:
application_data3["patent_issue_date"].min(skipna=True)

Timestamp('1998-06-07 00:00:00')

In [303]:
application_data3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11019954 entries, 2270746 to 14100377
Data columns (total 6 columns):
 #   Column              Dtype         
---  ------              -----         
 0   filing_date         datetime64[ns]
 1   examiner_full_name  object        
 2   examiner_art_unit   Int64         
 3   appl_status_desc    string        
 4   patent_number       object        
 5   patent_issue_date   datetime64[ns]
dtypes: Int64(1), datetime64[ns](2), object(2), string(1)
memory usage: 599.0+ MB


In [304]:
# I want to create a leniency measure for examiners and their tech/art unit. For this I have to inspect the "appl_status_desc" column
status_entries = application_data3["appl_status_desc"].dropna().unique()
for s in sorted(status_entries):
    print(s)

# As we can see, there are lot of status explanations. Among these some of them are application procedures or applicant action.

"Decision on Petition Denied, Reexam Request Denied, Terminated"
'Decision on Petition Denied, Reexam Request Denied, Terminated'
ABANDONED - RESTORED
AFTER APPEAL DECISION
AWAITING RESPONSE FOR INFORMALITY, FEE DEFICIENCY OR CRF ACTION
Abandoned -- After Examiner's Answer or Board of Appeals Decision
Abandoned -- Failure to Pay Issue Fee
Abandoned -- Failure to Respond to an Office Action
Abandoned -- File-Wrapper-Continuation Parent Application
Abandoned -- Incomplete (Filing Date Under Rule 53 (b) - PreExam)
Abandoned -- Incomplete Application (Pre-examination)
Abandonment for Failure to Correct Drawings/Oath/NonPub Request
Advisory Action Mailed
Amendment / Argument after Board of Appeals Decision
Amendment after notice of appeal
Appeal Brief (or Supplemental Brief) Entered and Forwarded to Examiner
Appeal Brief Filed (or Remand from Board) - Awaiting Examiner Action
Appeal Dismissed / Withdrawn
Appeal Ready for Review
Application Dispatched from Preexam, Not Yet Docketed
Applicati

In [305]:
# I only want the rows where the patents were granted or rejected by the examiner strictly
examiner_grant = [
"Patented Case",
"Patent Expired Due to NonPayment of Maintenance Fees Under 37 CFR 1.362",
"Patent Reinstated Following Decision on Maintenance Fee Petition Under 37 CFR 1.377 - 1.378", 
"Allowed -- Notice of Allowance Mailed -- Issue Revision Completed",
"Notice of Allowance Mailed -- Application Received in Office of Publications",
"Awaiting TC Resp, Issue Fee Payment Received",
"Awaiting TC Resp, Issue Fee Payment Verified",
"Publications -- Issue Fee Payment Received",
"Publications -- Issue Fee Payment Verified"
]

examiner_rejection = [
"Final Action Mailed",
"Final Rejection Mailed",
"Abandoned -- After Examiner's Answer or Board of Appeals Decision",
"Abandoned -- Failure to Pay Issue Fee",
"Abandoned -- Failure to Respond to an Office Action",
"Abandonment for Failure to Correct Drawings/Oath/NonPub Request",
"Abandoned -- File-Wrapper-Continuation Parent Application",
]

# Final Action Mailed may not be rejection in its strictest definition. It is only 22 entries nonetheless and I will do robustness check at the end by removing it.
# I do not consider 'abandoned' or 'patent expired' cases as rejection because they are driven by applicant behavior
# I have added patent expired due to nonpayment of maintenance fees. It implies the patent was granted by the examiner at some point. Deliberation is required on this.

examiner_grant_or_rejection = examiner_grant + examiner_rejection

# I modify the data set with the selections above
application_data4 = application_data3[application_data3["appl_status_desc"].isin(examiner_grant_or_rejection)].copy()
application_data4["appl_status_desc"].value_counts(dropna=False)

appl_status_desc
Patented Case                                                                   4032812
Abandoned -- Failure to Respond to an Office Action                             1804378
Patent Expired Due to NonPayment of Maintenance Fees Under 37 CFR 1.362         1569402
Abandoned -- Failure to Pay Issue Fee                                             74876
Notice of Allowance Mailed -- Application Received in Office of Publications      57070
Abandoned -- After Examiner's Answer or Board of Appeals Decision                 55286
Final Rejection Mailed                                                            52971
Publications -- Issue Fee Payment Verified                                        21017
Publications -- Issue Fee Payment Received                                         3257
Abandonment for Failure to Correct Drawings/Oath/NonPub Request                    1454
Awaiting TC Resp, Issue Fee Payment Verified                                        666
Awaiting TC Res

In [306]:
application_data4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7673311 entries, 2270746 to 13116421
Data columns (total 6 columns):
 #   Column              Dtype         
---  ------              -----         
 0   filing_date         datetime64[ns]
 1   examiner_full_name  object        
 2   examiner_art_unit   Int64         
 3   appl_status_desc    string        
 4   patent_number       object        
 5   patent_issue_date   datetime64[ns]
dtypes: Int64(1), datetime64[ns](2), object(2), string(1)
memory usage: 417.1+ MB


In [307]:
# Now I want to create a leniency measure for examiner art unit using final examiner decisions I have selected before
examiner_data_au = application_data4.dropna(subset=["examiner_art_unit"]).copy()

examiner_data_au["grants"] = examiner_data_au["appl_status_desc"].isin(examiner_grant)
examiner_data_au["rejections"] = examiner_data_au["appl_status_desc"].isin(examiner_rejection)

leniency_data_au = (examiner_data_au.groupby("examiner_art_unit", as_index=False).agg(n_grants=("grants","sum"),n_rejections=("rejections","sum")))

# I define a new variable to measure leniency for the art unit
leniency_data_au["leniency_art_unit"] = leniency_data_au["n_grants"]/(leniency_data_au["n_grants"]+leniency_data_au["n_rejections"])

In [308]:
# I do the same for tech center
#examiner_data_tc = application_data4.dropna(subset=["examiner_tech_center_number"]).copy()
#examiner_data_tc.sample(10)

#examiner_data_tc["grants"] = examiner_data_tc["appl_status_desc"].isin(examiner_grant)
#examiner_data_tc["rejections"] = examiner_data_tc["appl_status_desc"].isin(examiner_rejection)

#leniency_data_tc = (examiner_data_tc.groupby("examiner_tech_center_number", as_index=False).agg(n_grants=("grants","sum"),n_rejections=("rejections","sum")))

#leniency_data_tc["leniency_tech_center_number"] = leniency_data_tc["n_grants"]/(leniency_data_tc["n_grants"]+leniency_data_tc["n_rejections"])

In [309]:
leniency_data_au.sample(10)

,examiner_art_unit,n_grants,n_rejections,leniency_art_unit
32,1643,5635,4867,0.536564
263,2455,5232,1792,0.744875
329,2636,8561,1709,0.833593
745,4161,21,1,0.954545
353,2664,9617,1176,0.891040
523,2897,9986,1450,0.873207
501,2872,43915,10993,0.799792
264,2456,4302,2045,0.677801
43,1655,8327,6280,0.570069
125,1785,5263,2841,0.649432


In [310]:
mean = leniency_data_au["leniency_art_unit"].mean()
sd = leniency_data_au["leniency_art_unit"].std()
p25 = leniency_data_au["leniency_art_unit"].quantile(0.25)
p75 = leniency_data_au["leniency_art_unit"].quantile(0.75)
iqr = p75 - p25

print("mean",mean)
print("standard deviation", sd)
print("percentile25", p25)
print("percentile75", p75)
print("iqr", iqr)

mean 0.7743603638154885
standard deviation 0.15436678883109323
percentile25 0.6930097664623566
percentile75 0.8785060712512711
iqr 0.18549630478891455


In [311]:
#leniency_data_tc.sample(10)

In [312]:
#leniency_data_tc["leniency_tech_center_number"].mean()

In [313]:
# Now I want to match the leniency measures above with a copy of the application_data2.
application_data5 = application_data4.merge(leniency_data_au[["examiner_art_unit","leniency_art_unit"]], on="examiner_art_unit", how="left")
#application_data5 = application_data5.merge(leniency_data_tc[["examiner_tech_center_number","leniency_tech_center_number"]],on="examiner_tech_center_number", how="left")


In [314]:
# I check the merge and missing values
application_data5["leniency_art_unit"].isna().sum()


np.int64(9157)

In [315]:
missing_key = application_data5["examiner_art_unit"].isna().sum()
print("Rows with missing examiner_art_unit:", missing_key)

Rows with missing examiner_art_unit: 9157


In [316]:
au_in_apps = set(application_data5["examiner_art_unit"].dropna().unique())
au_in_len  = set(leniency_data_au["examiner_art_unit"].dropna().unique())

missing_aus = au_in_apps - au_in_len
print("Number of art units missing from leniency table:", len(missing_aus))
list(sorted(missing_aus))

Number of art units missing from leniency table: 0


[]

In [317]:
# leniency measures are biased upwards by definition. We are excluding ambigious status entries while including patent granted entries that have been through same procedures.
# I will create a new measure for grant prosperity. This is biased downwards by definition since I will include all the ambigious status entries.
# To do this I focus on the patent number

application_data4["patent_number"] = application_data4["patent_number"].astype("string")
application_data4["every_patent_number"] = application_data4["patent_number"].notna() & (application_data4["patent_number"].str.strip() !="")

eventual_leniency_art_unit = (application_data4.dropna(subset=["examiner_art_unit"]).groupby("examiner_art_unit", as_index=False)
                     .agg(n_apps=("every_patent_number","size"),n_granted=("every_patent_number","sum")))

eventual_leniency_art_unit["grant_propensity_art_unit"] = eventual_leniency_art_unit["n_granted"] / eventual_leniency_art_unit["n_apps"]

In [318]:
eventual_leniency_art_unit.sample(10)

,examiner_art_unit,n_apps,n_granted,grant_propensity_art_unit
270,2463,9329,7608,0.815521
15,1624,31807,23470,0.737888
335,2644,9063,7306,0.806135
68,1718,3570,1216,0.340616
406,2744,305,298,0.977049
108,1765,25850,19838,0.767427
326,2633,11722,10123,0.86359
228,2412,6112,5239,0.857166
55,1676,3072,2039,0.663737
422,2762,306,302,0.986928


In [319]:
mean = eventual_leniency_art_unit["grant_propensity_art_unit"].mean()
sd = eventual_leniency_art_unit["grant_propensity_art_unit"].std()
p25 = eventual_leniency_art_unit["grant_propensity_art_unit"].quantile(0.25)
p75 = eventual_leniency_art_unit["grant_propensity_art_unit"].quantile(0.75)
iqr = p75 - p25

print("mean",mean)
print("standard deviation", sd)
print("percentile25", p25)
print("percentile75", p75)
print("iqr", iqr)


mean 0.7621497110679955
standard deviation 0.15608069688033596
percentile25 0.6792534121327896
percentile75 0.8611301489232922
iqr 0.18187673679050265


In [320]:
# I do the same for the tech center only

#eventual_leniency_tech_center = (application_data4.dropna(subset=["examiner_tech_center_number"]).groupby("examiner_tech_center_number", as_index=False)
                     #.agg(n_apps=("every_patent_number","size"),n_granted=("every_patent_number","sum")))

#eventual_leniency_tech_center["grant_propensity_tech_center"] = eventual_leniency_tech_center["n_granted"] / eventual_leniency_tech_center["n_apps"]

In [321]:
#eventual_leniency_tech_center.sample(10)

In [322]:
#eventual_leniency_tech_center["grant_propensity_tech_center"].mean()

In [323]:
application_data6 = application_data5.merge(eventual_leniency_art_unit[["examiner_art_unit","grant_propensity_art_unit"]], on="examiner_art_unit", how="left")
#application_data6 = application_data6.merge(eventual_leniency_tech_center[["examiner_tech_center_number","grant_propensity_tech_center"]], on="examiner_tech_center_number", how="left")

In [324]:
application_data6.head()

,filing_date,examiner_full_name,examiner_art_unit,appl_status_desc,patent_number,patent_issue_date,leniency_art_unit,grant_propensity_art_unit
0,1997-12-01,"COLEMAN, BRENDA LIBBY",1624,Abandoned -- Failure to Respond to an Office A...,NaN,NaT,0.745025,0.737888
1,1997-12-01,"SELLERS, ROBERT E",1712,Abandoned -- Failure to Respond to an Office A...,NaN,NaT,0.647294,0.640682
2,1997-12-01,"MCCARRY JR, ROBERT J",3613,Patented Case,5979334,1999-11-09,0.968898,0.968898
3,1997-12-01,"KENNEDY, JAMES C",1764,Patent Expired Due to NonPayment of Maintenanc...,5948370,1999-09-07,0.698670,0.685368
4,1997-12-01,"SHACKELFORD, HEATHER CHUN",3671,Patent Expired Due to NonPayment of Maintenanc...,5918390,1999-07-06,0.775472,0.764996


In [325]:
stats = application_data6.copy()

stats["diff"] = stats["leniency_art_unit"]-stats["grant_propensity_art_unit"]
stats["abs_diff"] = stats["diff"].abs()
stats["pct_diff"] = stats["diff"] / stats["grant_propensity_art_unit"].replace(0, pd.NA)

stats.sample(10)
mean = stats["pct_diff"].mean()
min = stats["pct_diff"].min()
max = stats["pct_diff"].max()

print("mean:", mean)
print("min:", min)
print("max:", max)

# On average the difference between two measures is 1.6 percent for art units.

mean: 0.017824752572508942
min: -0.0034129692832763894
max: 0.3333333333333333


In [326]:
#stats = application_data6.copy()

#stats["diff"] = stats["leniency_tech_center_number"]-stats["grant_propensity_tech_center"]
#stats["abs_diff"] = stats["diff"].abs()
#stats["pct_diff"] = stats["diff"] / stats["grant_propensity_tech_center"].replace(0, pd.NA)

#stats.sample(10)
#mean = stats["pct_diff"].mean()
#min = stats["pct_diff"].min()
#max = stats["pct_diff"].max()

#print("mean:", mean)
#print("min:", min)
#print("max:", max)

# On average the difference between two measures is 1.6 percent for tech centers.

In [327]:
application_data6.to_csv("/Users/cantutuncu/Documents/Bocconi University/THESIS/Data/merged_data/application_data_final.csv", index = False)